In [ ]:
# Check if the path prepended prefixes have other ASes on their paths.

import pybgpstream
import pandas as pd
from multiprocessing import Pool, cpu_count

# Load DataFrame (assuming df is already loaded)
scrubber = "19551"  # Replace with the actual ASN you want to exclude
year = "2024"

df = pd.read_csv("../data/unique_optimized_provider_as"+scrubber+"_path_prepend_01_jan_"+year+".csv")
prefixes = df["prefix"].unique()

def has_other_upstream(pfx):
    """
    Check if a given prefix has an upstream ASN other than the scrubber.
    """
    stream = pybgpstream.BGPStream(
        from_time="2024-01-01 00:00:00 UTC",
        until_time="2024-01-01 00:00:00 UTC",
        record_type="ribs",   
        filter="prefix exact " + pfx + " and path !_" + scrubber + "_"
    )

    stream.set_data_interface_option("broker", "cache-dir", "/home/shyam/jupy/cache")

    for rec in stream.records():
        for elem in rec:
            if elem:
#                 print(f"Prefix with other upstream: {pfx}, and elem is {elem}")  # Print matching records
                return pfx  # Return prefix if another upstream exists
    return None  # Return None if no other upstream found


"""
Main function to run prefix checks in parallel.
"""
#     with Pool(processes=cpu_count()) as pool:
with Pool(processes=6) as pool:
    results = pool.map(has_other_upstream, prefixes)

 # Filter out None values (prefixes that had no other upstream)
valid_prefixes = [pfx for pfx in results if pfx]

# Count how many prefixes have other ASNs as upstream
count = len(valid_prefixes)

print(f"Total prefixes with other upstreams: {count}   out of {len(prefixes)} prefixes.")
print("List of prefixes with other upstreams:", valid_prefixes)


In [ ]:
# Other approach is to look into raw bgp data on that date and check AS paths using multi processing.
import bz2
import pandas as pd

scrubber = "32787"
year = "2024"

df = pd.read_csv('../data/unique_optimized_raw_01_jan_'+year+'.csv.bz2', compression="bz2")

df[["new_provider"]] = df.apply(determine_path_prepending_and_new_provider, axis=1)

# Define a function to determine other upstream
def determine_other_upstream(row):
   
    # Ensure as_path is a string and split into a list of ASNs
    as_path = list(map(int, str(row['as_path']).split()))
    if scrubber not in as_path:
        new_provider = 1 
    
    return pd.Series([new_provider])


In [ ]:
# For a list of given prefixes find how many of them have other upstream providers.
# It took a lot of time (it was not finished until 3 hrs. So, looking into pybgpstream seem to be a better idea.)

import pandas as pd
import multiprocessing as mp

def filter_single_prefix(args):
    """Filter the dataframe for a single prefix."""
    df_full, pfx, scrubber_asn = args

    # Select only rows where prefix matches
    df_filtered = df_full[df_full["prefix"] == pfx]

    # Ensure AS path column is properly formatted as strings
    df_filtered = df_filtered.dropna(subset=["as_path"])  # Remove NaN rows
    df_filtered["as_path"] = df_filtered["as_path"].astype(str).str.strip()

    # Regex to match whole ASNs correctly
    regex_pattern = fr'\b{scrubber_asn}\b'

    # Keep rows where scrubber_asn is NOT present
    df_cleaned = df_filtered[~df_filtered["as_path"].str.contains(regex_pattern, na=False, regex=True)]

    return df_cleaned
    
def filter_prefixes_parallel(df_full, prefixes, scrubber_asn):
    """Use multiprocessing to filter prefixes in parallel."""
    with mp.Pool(processes=mp.cpu_count()) as pool:
        results = pool.map(filter_single_prefix, [(df_full, pfx, scrubber_asn) for pfx in prefixes])

    # Combine results into a single DataFrame
    return pd.concat(results, ignore_index=True)

# Example usage
if __name__ == "__main__":
    df_full = df
    
    # ASN to exclude
    scrubber= "32787"
    
    # List of prefixes to check
    df_pfxs = pd.read_csv("unique_optimized_provider_as"+scrubber+"_path_prepend_01_jan_"+year+".csv")
    prefixes = df_pfxs["prefix"].unique()

    # Run in parallel
    result_df = filter_prefixes_parallel(df_full, prefixes, int(scrubber))

    print(result_df)


In [ ]:
# Other approach is to look into raw bgp data on that date and check AS paths using multi processing.
# Sample code

import pandas as pd
import multiprocessing as mp

def filter_single_prefix(args):
    """Filter the dataframe for a single prefix."""
    df_full, pfx, scrubber_asn = args

    # Select only rows where prefix matches
    df_filtered = df_full[df_full["prefix"] == pfx]

    # Ensure AS path column is properly formatted as strings
    df_filtered = df_filtered.dropna(subset=["as_path"])  # Remove NaN rows
    df_filtered["as_path"] = df_filtered["as_path"].astype(str).str.strip()

    # Regex to match whole ASNs correctly
    regex_pattern = fr'\b{scrubber_asn}\b'

    # Keep rows where scrubber_asn is NOT present
    df_cleaned = df_filtered[~df_filtered["as_path"].str.contains(regex_pattern, na=False, regex=True)]

    return df_cleaned

def filter_prefixes_parallel(df_full, prefixes, scrubber_asn):
    """Use multiprocessing to filter prefixes in parallel."""
    with mp.Pool(processes=mp.cpu_count()) as pool:
        results = pool.map(filter_single_prefix, [(df_full, pfx, scrubber_asn) for pfx in prefixes])

    # Combine results into a single DataFrame
    return pd.concat(results, ignore_index=True)

# Example usage
if __name__ == "__main__":
    # Sample dataframe
    data = {
        "prefix": [
            "192.168.1.0/24",
            "10.0.0.0/8",
            "172.16.0.0/12",
            "159.50.124.0/24"  # Problematic AS Path
        ],
        "as_path": [
            "123 456 789",
            "100 200 300",
            "123 888 999",
            "5392 3356 25215 25215 25215 25215"  # Should not be removed
        ]
    }

    df_full = pd.DataFrame(data)

    # List of prefixes to check
    prefixes = ["192.168.1.0/24", "10.0.0.0/8", "159.50.124.0/24"]

    # ASN to exclude
    scrubber_asn = 32787

    # Run in parallel
    result_df = filter_prefixes_parallel(df_full, prefixes, scrubber_asn)

    print(result_df)


